# Lab2 - Analisis de Imagen, Texto y Audio
Ejecuta cada celda en orden.

In [ ]:
# CELDA 1 - Instalar dependencias
# Esto puede tardar varios minutos la primera vez
!pip install openai-whisper transformers torch torchvision torchaudio Pillow pysentimiento cohere -q

In [ ]:
# CELDA 2 - Importar librerias y cargar modelos
import whisper                                   # Transcripcion de audio a texto
import torch                                     # Framework de deep learning
import io                                        # Manejo de bytes en memoria
import os                                        # Lee la variable de entorno COHERE_API_KEY
import time                                      # Mide el tiempo de cada llamada a Cohere
import cohere                                    # SDK oficial de Cohere para llamar a la API de Command R
from PIL import Image                            # Procesamiento de imagenes
from transformers import pipeline, CLIPProcessor, CLIPModel  # Modelos de HuggingFace
from google.colab import files, userdata         # Sube archivos y lee secretos desde Colab

device = "cuda" if torch.cuda.is_available() else "cpu"  # Usa GPU de Colab si esta disponible
print(f"Usando dispositivo: {device}")

print("Cargando Whisper...")
model_whisper = whisper.load_model("base", device=device)  # Modelo de transcripcion de audio

print("Cargando modelo de sentimiento...")
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis",  # Modelo entrenado en español nativo
    device=device
)

print("Cargando CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)  # Compara imagen y texto
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")     # Preprocesador de CLIP

print("Modelos listos.")

In [ ]:
# CELDA 3 - Subir archivos
# Se abrira un selector de archivos para cada uno

print("Sube tu imagen (JPG, PNG):")
imagen_subida = files.upload()                   # Abre selector para subir la imagen
imagen_nombre = list(imagen_subida.keys())[0]    # Obtiene el nombre del archivo subido
image_bytes = imagen_subida[imagen_nombre]       # Obtiene los bytes de la imagen

print("Sube tu audio (WAV):")
audio_subido = files.upload()                    # Abre selector para subir el audio
audio_nombre = list(audio_subido.keys())[0]      # Obtiene el nombre del archivo subido
audio_bytes = audio_subido[audio_nombre]         # Obtiene los bytes del audio

# Texto escrito por el usuario (edita este valor)
text_input = "Teléfono inteligente moderno con la pantalla severamente rota, con un patrón de grietas en forma de telaraña que se extiende por toda la superficie de la pantalla"  # Cambia este texto antes de ejecutar

print(f"Imagen cargada: {imagen_nombre}")
print(f"Audio cargado: {audio_nombre}")
print(f"Texto: {text_input}")

In [ ]:
# CELDA 4 - Pasos A, B y C: transcripcion, sentimiento y validacion visual

# --- A. TRANSCRIPCION DE AUDIO ---
with open("temp_audio.wav", "wb") as f:          # Whisper requiere archivo en disco, no bytes en memoria
    f.write(audio_bytes)

audio_result = model_whisper.transcribe("temp_audio.wav")  # Transcribe el audio a texto
transcribed_text = audio_result['text'].strip()            # Extrae el texto y elimina espacios sobrantes

# --- B. ANALISIS DE SENTIMIENTO ---
combined_text = f"{text_input}. {transcribed_text}"  # Une texto escrito y audio transcrito
sentiment = sentiment_analyzer(combined_text)[0]     # Devuelve {"label": "NEG/POS/NEU", "score": 0.0-1.0}

# --- C. VALIDACION IMAGEN VS TEXTO (CLIP) ---
image = Image.open(io.BytesIO(image_bytes)).convert("RGB")  # Decodifica bytes de la imagen a RGB

inputs = clip_processor(
    text=[transcribed_text],                     # Solo el audio transcrito se compara contra la imagen
    images=image,
    return_tensors="pt",
    padding=True
).to(device)

outputs = clip_model(**inputs)                   # Calcula similitud entre imagen y texto
relevance_score = outputs.logits_per_image[0][0].item()  # Score raw sin softmax

# Umbral calibrado: imagen rota + texto rota = 24.14, imagen buena + texto rota = 20.98
COINCIDENCIA_UMBRAL = 22.5                       # 22.5 separa ambos casos con margen suficiente
alerta = relevance_score < COINCIDENCIA_UMBRAL   # True si la imagen no coincide con el texto
mensaje_alerta = (
    "ALERTA: La imagen no parece representar lo que se describe en el texto o el audio."
    if alerta else
    "La imagen coincide con el contexto."
)

print("=" * 50)
print(f"Transcripcion : {transcribed_text}")
print(f"Sentimiento   : {sentiment['label']} (confianza: {sentiment['score']:.2f})")
print(f"Score CLIP    : {round(relevance_score, 4)}")
print(f"Discrepancia  : {alerta}")
print(f"Validacion    : {mensaje_alerta}")
print("=" * 50)

In [ ]:
# CELDA 5 - Configurar clave de API de Cohere
# Obtén tu clave gratis en https://dashboard.cohere.com/api-keys

# Opcion A: usar Colab Secrets (recomendado)
# Ve a Herramientas > Secrets, agrega COHERE_API_KEY con tu clave
try:
    os.environ["COHERE_API_KEY"] = userdata.get("COHERE_API_KEY")  # Lee el secreto de Colab
    print("Clave cargada desde Colab Secrets.")
except Exception:
    # Opcion B: pegar la clave directamente (menos seguro)
    os.environ["COHERE_API_KEY"] = "..."         # Reemplaza con tu clave real
    print("Clave cargada manualmente.")

cohere_client = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])  # Cliente V2 de Cohere
print("Cohere listo.")

In [ ]:
# CELDA 6 - Paso D: Dictamen final con tres tecnicas de prompting usando Cohere

# --- TECNICA 1: ZERO-SHOT ---
# Sin ejemplos: Cohere infiere el formato solo a partir de la instruccion
PROMPT_ZERO_SHOT = """
Eres un auditor de reclamos de productos dañados.

Datos del analisis automatico:
- Sentimiento del reclamo: {sentimiento_label} (confianza: {sentimiento_score:.2f})
- Transcripcion del audio: {transcripcion}
- Imagen coincide con el reclamo: {coincide}
- Score de coincidencia visual: {score}

Emite un dictamen breve: indica SI o NO se debe aprobar el reembolso y justifica en una sola oracion.
"""

# --- TECNICA 2: FEW-SHOT ---
# Con 3 ejemplos reales: Cohere aprende el patron de formato y criterio esperado
PROMPT_FEW_SHOT = """
Eres un auditor de reclamos de productos dañados.

Ejemplos de dictamenes anteriores:

Caso 1:
- Sentimiento: NEG (0.91) | Transcripcion: "el telefono llego con la pantalla rota"
- Imagen coincide: SI | Score: 24.14
- Dictamen: SI se aprueba el reembolso. La imagen confirma el daño descrito y el reclamo es genuinamente negativo.

Caso 2:
- Sentimiento: NEG (0.78) | Transcripcion: "el cargador no funciona"
- Imagen coincide: NO | Score: 20.98
- Dictamen: NO se aprueba el reembolso. La imagen no corresponde al daño descrito, lo que sugiere inconsistencia en el reclamo.

Caso 3:
- Sentimiento: POS (0.85) | Transcripcion: "el producto llego perfecto"
- Imagen coincide: SI | Score: 23.50
- Dictamen: NO se aprueba el reembolso. El reclamo indica satisfaccion con el producto, no hay daño que compensar.

Ahora analiza este caso:
- Sentimiento: {sentimiento_label} ({sentimiento_score:.2f}) | Transcripcion: "{transcripcion}"
- Imagen coincide: {coincide} | Score: {score}

Dictamen:"""

# --- TECNICA 3: CHAIN-OF-THOUGHT ---
# Razonamiento paso a paso: reduce errores en casos ambiguos o con señales contradictorias
PROMPT_COT = """
Eres un auditor de reclamos de productos dañados.

Datos del analisis automatico:
- Sentimiento del reclamo: {sentimiento_label} (confianza: {sentimiento_score:.2f})
- Transcripcion del audio: {transcripcion}
- Imagen coincide con el reclamo: {coincide}
- Score de coincidencia visual: {score}

Razona paso a paso antes de concluir:
1. ¿El sentimiento indica un reclamo genuino?
2. ¿La imagen es coherente con lo que describe el audio?
3. ¿Hay señales de inconsistencia o posible fraude?
4. Conclusion final: SI o NO aprobar el reembolso en una sola oracion.
"""

# Datos comunes que se inyectan en los tres prompts
datos = dict(
    sentimiento_label=sentiment["label"],
    sentimiento_score=sentiment["score"],
    transcripcion=transcribed_text,
    coincide="NO" if alerta else "SI",
    score=round(relevance_score, 4)
)

def llamar_cohere(prompt):                       # Llama a Cohere y retorna (texto, segundos_transcurridos)
    inicio = time.time()
    response = cohere_client.chat(
        model="command-a-03-2025",                  # Usando 'command-a-03-2025' como un modelo actualizado y disponible
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512                           # CoT puede necesitar mas tokens que zero-shot
    )
    duracion = round(time.time() - inicio, 2)
    return response.message.content[0].text, duracion # Corrected: Accessing text via message.content[0].text

print("Llamando a Cohere con las tres tecnicas...\n")

texto_zs,  tiempo_zs  = llamar_cohere(PROMPT_ZERO_SHOT.format(**datos))  # Tecnica 1
texto_fs,  tiempo_fs  = llamar_cohere(PROMPT_FEW_SHOT.format(**datos))   # Tecnica 2
texto_cot, tiempo_cot = llamar_cohere(PROMPT_COT.format(**datos))        # Tecnica 3

# --- RESULTADOS Y METRICAS ---
print("=" * 60)
print("TECNICA 1 — ZERO-SHOT")
print(f"Tiempo: {tiempo_zs}s")
print(f"Dictamen:\n{texto_zs}")

print("\n" + "=" * 60)
print("TECNICA 2 — FEW-SHOT")
print(f"Tiempo: {tiempo_fs}s")
print(f"Dictamen:\n{texto_fs}")

print("\n" + "=" * 60)
print("TECNICA 3 — CHAIN-OF-THOUGHT")
print(f"Tiempo: {tiempo_cot}s")
print(f"Dictamen:\n{texto_cot}")

print("\n" + "=" * 60)
print("RESUMEN DE TIEMPOS")
print(f"  Zero-Shot        : {tiempo_zs}s")
print(f"  Few-Shot         : {tiempo_fs}s")
print(f"  Chain-of-Thought : {tiempo_cot}s")
print(f"  Total            : {round(tiempo_zs + tiempo_fs + tiempo_cot, 2)}s")
print("=" * 60)